# LeetCode #109 — Convert Sorted List to Binary Search Tree: chọn giữa làm gốc, ba cách

**Mức độ:** Medium · **Chủ đề:** Linked List, Divide and Conquer, Tree, Binary Search Tree, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def sortedListToBST(self, head: Optional[ListNode]) -> Optional[TreeNode]:
```

Mục tiêu của notebook:

1. Ý tưởng chính: phần tử **giữa** làm gốc → hai nửa có kích thước gần bằng nhau → cây **cân bằng**.
2. Cách 1: đổi sang mảng (dễ, `O(n)` bộ nhớ phụ).
3. Cách 2: tìm giữa bằng **con trỏ nhanh/chậm** (`O(n log n)`).
4. Cách 3: **mô phỏng inorder** — dựng cây trong khi duyệt danh sách một lần (`O(n)`, đẹp nhất).

# 1. Bài toán là gì?

Cho `head` của một danh sách liên kết đơn **sắp xếp tăng dần**, chuyển nó thành một BST **cân bằng chiều cao** (height-balanced): với mọi nút, chiều cao hai cây con chênh nhau **không quá 1**.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 2 * 10^4]`.
- `-10^5 <= Node.val <= 10^5`

---

## Ví dụ

```text
head = -10 -> -3 -> 0 -> 5 -> 9

Một đáp án:          0
                    / \
                  -3   9          [0, -3, 9, -10, null, 5]
                  /   /
                -10  5

head = []  ->  []
```

Đề chấp nhận **mọi** BST cân bằng hợp lệ; nên khi kiểm tra ta không so với một cây cụ thể mà kiểm tra **tính chất**.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Danh sách liên kết đơn (singly linked list)
2. Cây nhị phân tìm kiếm (BST)
3. Chia để trị (divide and conquer)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [2]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [3]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [4]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [5]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.2 Cây nhị phân tìm kiếm (BST)

### BST là gì?

Cây nhị phân tìm kiếm (Binary Search Tree) là cây nhị phân thỏa, **với mọi nút**:

- mọi giá trị trong cây con **trái** `<` giá trị nút,
- mọi giá trị trong cây con **phải** `>` giá trị nút.

```text
        8
       / \
      3   10
     / \    \
    1   6    14
       / \   /
      4   7 13
```

Chú ý: điều kiện áp dụng cho **toàn bộ** cây con, không chỉ con trực tiếp (ví dụ `7` nằm bên trái `8` nên phải `< 8`).

### Ba tính chất quan trọng

1. **Tìm kiếm** giống tìm nhị phân: so với nút, nhỏ hơn thì sang trái, lớn hơn thì sang phải → `O(h)` (h = chiều cao).
2. **Duyệt inorder cho dãy tăng dần**: `1 3 4 6 7 8 10 13 14`.
3. Cây **cân bằng** có `h ≈ log₂ n`; cây lệch có `h = n` (tệ như danh sách).

In [6]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def bst_chen(root, x):
    if root is None:
        return TreeNode(x)
    if x < root.val:
        root.left = bst_chen(root.left, x)
    else:
        root.right = bst_chen(root.right, x)
    return root

def bst_tim(root, x):
    buoc = 0
    while root:
        buoc += 1
        if x == root.val:
            return True, buoc
        root = root.left if x < root.val else root.right
    return False, buoc

def bst_inorder(node, out):
    if node:
        bst_inorder(node.left, out); out.append(node.val); bst_inorder(node.right, out)

r = None
for x in [8, 3, 10, 1, 6, 14, 4, 7, 13]:
    r = bst_chen(r, x)
out = []; bst_inorder(r, out)
print("inorder:", out)
print("tìm 7:", bst_tim(r, 7), "| tìm 5:", bst_tim(r, 5), "(đã tìm, số bước)")

inorder: [1, 3, 4, 6, 7, 8, 10, 13, 14]
tìm 7: (True, 4) | tìm 5: (False, 4) (đã tìm, số bước)


### Dựng BST cân bằng từ dãy đã sắp xếp

Lấy phần tử **giữa** làm gốc, nửa trái dựng cây con trái, nửa phải dựng cây con phải (chia để trị). Chiều cao chỉ `O(log n)`.

In [7]:
def bst_tu_mang(a, lo=0, hi=None):
    if hi is None:
        hi = len(a) - 1
    if lo > hi:
        return None
    mid = (lo + hi) // 2
    return TreeNode(a[mid], bst_tu_mang(a, lo, mid - 1), bst_tu_mang(a, mid + 1, hi))

def cao(n):
    return 0 if n is None else 1 + max(cao(n.left), cao(n.right))

lech = None
for x in range(1, 16):
    lech = bst_chen(lech, x)                  # chèn tăng dần → cây lệch
print("15 số chèn tăng dần: cao", cao(lech), "| dựng từ giữa: cao", cao(bst_tu_mang(list(range(1, 16)))))

15 số chèn tăng dần: cao 15 | dựng từ giữa: cao 4


### Kiểm tra một cây có phải BST

Sai lầm phổ biến: chỉ so mỗi nút với **con trực tiếp**. Đúng: truyền xuống **khoảng cho phép** `(lo, hi)`; sang trái thì cận trên thành giá trị nút, sang phải thì cận dưới thành giá trị nút. (Hoặc: inorder phải tăng ngặt.)

In [8]:
def la_bst(node, lo=float("-inf"), hi=float("inf")):
    if node is None:
        return True
    if not (lo < node.val < hi):
        return False
    return la_bst(node.left, lo, node.val) and la_bst(node.right, node.val, hi)

def chi_so_con(node):                          # cách SAI
    if node is None:
        return True
    if node.left and node.left.val >= node.val: return False
    if node.right and node.right.val <= node.val: return False
    return chi_so_con(node.left) and chi_so_con(node.right)

bay = TreeNode(5, TreeNode(1), TreeNode(7, TreeNode(4), TreeNode(8)))   # 4 nằm bên phải 5 → không phải BST
print("đúng:", la_bst(bay), "| cách chỉ so con trực tiếp (sai):", chi_so_con(bay))

đúng: False | cách chỉ so con trực tiếp (sai): True


## 2.3 Chia để trị (divide and conquer)

### Ý tưởng

1. **Chia** bài toán thành các bài con **cùng dạng**, nhỏ hơn (thường là hai nửa).
2. **Trị**: giải các bài con bằng đệ quy (bài đủ nhỏ thì giải trực tiếp).
3. **Kết hợp** kết quả các bài con thành kết quả bài lớn.

Khác DP: các bài con **không chồng lấp** (mỗi phần dữ liệu thuộc đúng một bài con).

```text
            [8 phần tử]                      tầng 0: 1 bài, tổng n việc
         [4]          [4]                    tầng 1: 2 bài, tổng n việc
       [2]  [2]     [2]  [2]                 tầng 2: 4 bài, tổng n việc
      [1][1][1][1] [1][1][1][1]              log₂ n tầng  →  O(n log n)
```

In [9]:
def max_chia_de_tri(a, lo, hi, sau=0):
    """Phần tử lớn nhất của a[lo..hi]."""
    if lo == hi:
        return a[lo]                                   # bài đủ nhỏ
    mid = (lo + hi) // 2
    trai = max_chia_de_tri(a, lo, mid, sau + 1)        # trị nửa trái
    phai = max_chia_de_tri(a, mid + 1, hi, sau + 1)    # trị nửa phải
    kq = max(trai, phai)                               # kết hợp
    print("    " * sau + f"a[{lo}..{hi}] = {a[lo:hi + 1]} → max {kq}")
    return kq

_ = max_chia_de_tri([3, 8, 1, 9, 4, 7, 2, 6], 0, 7)

        a[0..1] = [3, 8] → max 8
        a[2..3] = [1, 9] → max 9
    a[0..3] = [3, 8, 1, 9] → max 9
        a[4..5] = [4, 7] → max 7
        a[6..7] = [2, 6] → max 6
    a[4..7] = [4, 7, 2, 6] → max 7
a[0..7] = [3, 8, 1, 9, 4, 7, 2, 6] → max 9


### Các ví dụ quen thuộc

| Bài | Chia | Kết hợp | Độ phức tạp |
|---|---|---|---|
| Tìm nhị phân | bỏ một nửa | không cần | `O(log n)` |
| Merge sort | hai nửa | trộn `O(n)` | `O(n log n)` |
| Lũy thừa nhanh `x^n` | `x^(n/2)` | bình phương | `O(log n)` |
| Dựng cây cân bằng từ mảng sắp xếp | phần tử giữa làm gốc | gắn hai cây con | `O(n)` |
| Dựng cây từ hai cách duyệt | tách theo vị trí gốc | gắn hai cây con | `O(n)` với hash map |

In [10]:
def luy_thua_nhanh(x, n, sau=0):
    if n == 0:
        return 1
    nua = luy_thua_nhanh(x, n // 2, sau + 1)
    kq = nua * nua * (x if n % 2 else 1)
    print("    " * sau + f"{x}^{n} = ({x}^{n // 2})²{' · ' + str(x) if n % 2 else ''} = {kq}")
    return kq

_ = luy_thua_nhanh(2, 10)

            2^1 = (2^0)² · 2 = 2
        2^2 = (2^1)² = 4
    2^5 = (2^2)² · 2 = 32
2^10 = (2^5)² = 1024


Lỗi hay gặp: chia không đều hoặc không nhỏ đi (`mid` tính sai → đệ quy vô hạn); tạo bản sao `a[:mid]` ở mỗi tầng (tốn thêm `O(n log n)` bộ nhớ — truyền chỉ số `lo, hi` thay vì cắt list); quên trường hợp đoạn rỗng.

## 2.4 Áp dụng vào bài này

Dựng **BST cân bằng** từ dãy đã sắp xếp = lấy phần tử **giữa** làm gốc, **chia để trị** hai nửa. Khó ở chỗ dữ liệu là **danh sách liên kết** (không truy cập `a[mid]` được): tìm giữa bằng **nhanh/chậm**, hoặc chép ra mảng, hoặc mô phỏng duyệt inorder để đi dọc danh sách đúng một lần.

# 3. Nền tảng

In [11]:
from typing import List, Optional
from collections import deque
import random, sys
sys.setrecursionlimit(10000)


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_list(values):
    dummy = tail = ListNode()
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def serialize(root):
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node:
            out.append(node.val); q.append(node.left); q.append(node.right)
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out


def inorder_of(root):
    return inorder_of(root.left) + [root.val] + inorder_of(root.right) if root else []


def height_balanced(root):
    """Trả (cân bằng?, chiều cao)."""
    if not root:
        return True, 0
    lb, lh = height_balanced(root.left)
    rb, rh = height_balanced(root.right)
    return lb and rb and abs(lh - rh) <= 1, 1 + max(lh, rh)

Một cây là đáp án hợp lệ khi: (1) inorder bằng đúng dãy giá trị của danh sách (vì là BST trên đúng các phần tử đó), và (2) cân bằng chiều cao.

# 4. Ý tưởng chính: giữa làm gốc

Với dãy đã sắp xếp `a[lo..hi]`, chọn `mid = (lo + hi) // 2` làm gốc:

- Mọi phần tử bên trái `< a[mid]` → cây con trái từ `a[lo..mid−1]`.
- Mọi phần tử bên phải `> a[mid]` → cây con phải từ `a[mid+1..hi]`.
- Hai nửa có kích thước chênh nhau tối đa 1 → đệ quy cho ra cây cân bằng.

Trên **mảng**, lấy phần tử giữa là `O(1)`. Trên **danh sách liên kết**, không truy cập ngẫu nhiên được — đó là toàn bộ cái khó của bài này.

# 5. Cách 1: Đổi sang mảng

In [12]:
def to_bst_array(head: Optional[ListNode]) -> Optional[TreeNode]:
    vals = []
    while head:
        vals.append(head.val)
        head = head.next
    def build(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        return TreeNode(vals[mid], build(lo, mid - 1), build(mid + 1, hi))
    return build(0, len(vals) - 1)

r = to_bst_array(build_list([-10, -3, 0, 5, 9]))
print(serialize(r), "| inorder:", inorder_of(r), "| cân bằng:", height_balanced(r)[0])

[0, -10, 5, None, -3, None, 9] | inorder: [-10, -3, 0, 5, 9] | cân bằng: True


`O(n)` thời gian, `O(n)` bộ nhớ phụ cho mảng. Thường là lời giải được chấp nhận trong phỏng vấn — nhưng người hỏi có thể yêu cầu **không** chép ra mảng.

# 6. Cách 2: Con trỏ nhanh/chậm tìm giữa

Trên đoạn `[head, tail)` (tail **không** thuộc đoạn): `slow` đi 1 bước, `fast` đi 2 bước; khi `fast` chạm `tail`, `slow` ở giữa.

In [13]:
def to_bst_fast_slow(head: Optional[ListNode]) -> Optional[TreeNode]:
    def build(start, end):                    # đoạn [start, end)
        if start is end:
            return None
        slow = fast = start
        while fast is not end and fast.next is not end:
            slow = slow.next
            fast = fast.next.next
        root = TreeNode(slow.val)
        root.left = build(start, slow)
        root.right = build(slow.next, end)
        return root
    return build(head, None)

r = to_bst_fast_slow(build_list([-10, -3, 0, 5, 9]))
print(serialize(r), "| inorder:", inorder_of(r), "| cân bằng:", height_balanced(r)[0])

[0, -3, 9, -10, None, 5] | inorder: [-10, -3, 0, 5, 9] | cân bằng: True


Mỗi tầng đệ quy duyệt tổng cộng `O(n)` nút để tìm giữa; có `O(log n)` tầng → `O(n log n)` thời gian, `O(log n)` bộ nhớ (đệ quy). Dùng **khoảng nửa mở** `[start, end)` nên không phải cắt danh sách.

# 7. Cách 3: Mô phỏng inorder — `O(n)`

Quan sát: inorder của BST kết quả **chính là** thứ tự của danh sách. Vậy ta dựng cây **theo thứ tự inorder**:

1. Đếm `n` = số nút.
2. `build(lo, hi)` dựng cây cho các vị trí `lo..hi`:
   - dựng cây trái `build(lo, mid−1)` **trước** — trong lúc đó con trỏ danh sách tiến qua đúng `mid − lo` nút;
   - lúc này con trỏ đang ở phần tử thứ `mid` → tạo gốc, tiến con trỏ một bước;
   - dựng cây phải `build(mid+1, hi)`.

Con trỏ danh sách chỉ đi **một lần** từ đầu đến cuối → `O(n)`.

In [14]:
def to_bst_inorder(head: Optional[ListNode]) -> Optional[TreeNode]:
    n, node = 0, head
    while node:
        n += 1
        node = node.next
    cur = head

    def build(lo, hi):
        nonlocal cur
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        left = build(lo, mid - 1)
        root = TreeNode(cur.val)
        cur = cur.next
        root.left = left
        root.right = build(mid + 1, hi)
        return root

    return build(0, n - 1)

r = to_bst_inorder(build_list([-10, -3, 0, 5, 9]))
print(serialize(r), "| inorder:", inorder_of(r), "| cân bằng:", height_balanced(r)[0])

[0, -10, 5, None, -3, None, 9] | inorder: [-10, -3, 0, 5, 9] | cân bằng: True


# 8. Mô phỏng cách 3

In [15]:
def trace_inorder_build(values):
    head = build_list(values)
    cur = head
    def build(lo, hi, depth):
        nonlocal cur
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        print("  " * depth + f"build({lo}, {hi}): mid = {mid}, dựng trái trước")
        left = build(lo, mid - 1, depth + 1)
        print("  " * depth + f"-> con trỏ đang ở {cur.val}: tạo gốc {cur.val} cho đoạn [{lo}, {hi}]")
        root = TreeNode(cur.val, left)
        cur = cur.next
        root.right = build(mid + 1, hi, depth + 1)
        return root
    root = build(0, len(values) - 1, 0)
    print("Cây:", serialize(root))
    return root

_ = trace_inorder_build([-10, -3, 0, 5, 9])

build(0, 4): mid = 2, dựng trái trước
  build(0, 1): mid = 0, dựng trái trước
  -> con trỏ đang ở -10: tạo gốc -10 cho đoạn [0, 1]
    build(1, 1): mid = 1, dựng trái trước
    -> con trỏ đang ở -3: tạo gốc -3 cho đoạn [1, 1]
-> con trỏ đang ở 0: tạo gốc 0 cho đoạn [0, 4]
  build(3, 4): mid = 3, dựng trái trước
  -> con trỏ đang ở 5: tạo gốc 5 cho đoạn [3, 4]
    build(4, 4): mid = 4, dựng trái trước
    -> con trỏ đang ở 9: tạo gốc 9 cho đoạn [4, 4]
Cây: [0, -10, 5, None, -3, None, 9]


Các dòng "tạo gốc" xuất hiện theo đúng thứ tự `-10, -3, 0, 5, 9` — thứ tự inorder.

# 9. Phiên bản đúng format LeetCode

In [16]:
class Solution:
    def sortedListToBST(self, head: Optional[ListNode]) -> Optional[TreeNode]:
        n, node = 0, head
        while node:
            n += 1
            node = node.next
        cur = head

        def build(lo: int, hi: int) -> Optional[TreeNode]:
            nonlocal cur
            if lo > hi:
                return None
            mid = (lo + hi) // 2
            left = build(lo, mid - 1)
            root = TreeNode(cur.val, left)
            cur = cur.next
            root.right = build(mid + 1, hi)
            return root

        return build(0, n - 1)

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS  {actual}")


def valid_answer(values, root):
    return inorder_of(root) == values and height_balanced(root)[0]


sol = Solution()
check("Ví dụ 1 hợp lệ", valid_answer([-10, -3, 0, 5, 9], sol.sortedListToBST(build_list([-10, -3, 0, 5, 9]))), True)
check("Ví dụ 1 (cây)", serialize(sol.sortedListToBST(build_list([-10, -3, 0, 5, 9]))), [0, -10, 5, None, -3, None, 9])
check("Rỗng", serialize(sol.sortedListToBST(build_list([]))), [])
check("Một nút", serialize(sol.sortedListToBST(build_list([1]))), [1])
check("Hai nút", valid_answer([1, 3], sol.sortedListToBST(build_list([1, 3]))), True)

random.seed(109)
for _ in range(2000):
    values = sorted(random.sample(range(-100, 100), random.randint(0, 40)))
    for f in (sol.sortedListToBST, to_bst_array, to_bst_fast_slow, to_bst_inorder):
        assert valid_answer(values, f(build_list(values)))
big = list(range(20000))
check("20000 nút", valid_answer(big, sol.sortedListToBST(build_list(big))), True)
print("2000 danh sách ngẫu nhiên: 4 cách đều cho BST cân bằng đúng")

Ví dụ 1 hợp lệ         PASS  True
Ví dụ 1 (cây)          PASS  [0, -10, 5, None, -3, None, 9]
Rỗng                   PASS  []
Một nút                PASS  [1]
Hai nút                PASS  True


20000 nút              PASS  True
2000 danh sách ngẫu nhiên: 4 cách đều cho BST cân bằng đúng


Ví dụ 1 của bản LeetCode cho cây `[0, -10, 5, null, -3, null, 9]` — **khác** hình trong đề (`[0, -3, 9, -10, null, 5]`) vì ta chọn `mid = (lo + hi) // 2` (giữa lệch trái). Cả hai đều hợp lệ.

# 10. Vì sao đúng? Độ phức tạp?

**BST:** gốc là `a[mid]`, cây trái dựng từ các phần tử nhỏ hơn, cây phải từ các phần tử lớn hơn — đệ quy.

**Cân bằng:** hai đoạn con có kích thước `⌊(k−1)/2⌋` và `⌈(k−1)/2⌉`, chênh tối đa 1. Bằng quy nạp, cây dựng từ `k` phần tử theo cách này có chiều cao `⌈log2(k+1)⌉` — một hàm tăng theo `k`, nên hai cây con chênh nhau tối đa 1 về chiều cao.

**Cách 3:** `build` thăm các vị trí theo đúng thứ tự inorder (trái → gốc → phải), và con trỏ `cur` tiến đúng một bước mỗi khi tạo một nút — nên nút thứ `i` của cây (theo inorder) nhận giá trị thứ `i` của danh sách.

| Cách | Time | Bộ nhớ phụ |
|---|---|---|
| Mảng | `O(n)` | `O(n)` |
| Nhanh/chậm | `O(n log n)` | `O(log n)` |
| Mô phỏng inorder | `O(n)` | `O(log n)` |

# 11. Những lỗi dễ mắc

## Lỗi 1: Dựng cây theo thứ tự chèn (không cân bằng)

Chèn lần lượt các phần tử đã sắp xếp vào BST → cây suy biến thành một **chuỗi**.

In [18]:
def wrong_insert_sorted(head: Optional[ListNode]) -> Optional[TreeNode]:
    root = None
    def insert(node, v):
        if not node:
            return TreeNode(v)
        if v < node.val:
            node.left = insert(node.left, v)
        else:
            node.right = insert(node.right, v)
        return node
    while head:
        root = insert(root, head.val)              # SAI: dãy tăng -> chuỗi lệch phải
        head = head.next
    return root

r = wrong_insert_sorted(build_list([-10, -3, 0, 5, 9]))
print("Sai :", serialize(r), "| cân bằng?", height_balanced(r)[0], "| chiều cao", height_balanced(r)[1])

Sai : [-10, None, -3, None, 0, None, 5, None, 9] | cân bằng? False | chiều cao 5


## Lỗi 2: Cách 3 tạo gốc **trước** khi dựng cây trái

In [19]:
def wrong_root_first(head: Optional[ListNode]) -> Optional[TreeNode]:
    n, node = 0, head
    while node:
        n += 1; node = node.next
    cur = head
    def build(lo, hi):
        nonlocal cur
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        root = TreeNode(cur.val)                   # SAI: con trỏ chưa tới phần tử mid
        cur = cur.next
        root.left = build(lo, mid - 1)
        root.right = build(mid + 1, hi)
        return root
    return build(0, n - 1)

r = wrong_root_first(build_list([-10, -3, 0, 5, 9]))
print("Sai :", serialize(r), "| inorder:", inorder_of(r), "-> không còn là BST")

Sai : [-10, -3, 5, None, 0, None, 9] | inorder: [-3, 0, -10, 5, 9] -> không còn là BST


Tạo gốc trước rồi trái/phải là thứ tự **preorder** — giá trị bị gán theo preorder nên inorder không còn tăng dần.

## Lỗi 3: Nhanh/chậm không dừng đúng ở `end`

Nếu so `fast.next` với `None` thay vì với `end`, `slow` luôn là giữa của **cả phần còn lại tới cuối danh sách**, không phải giữa đoạn `[start, end)`. Ở nửa trái, `slow` có thể vượt ra ngoài đoạn — rồi `build(start, slow)` lại gọi đúng đoạn cũ, lặp vô hạn.

In [20]:
def wrong_fast_slow(head):
    def build(start, end):
        if start is end:
            return None
        slow = fast = start
        while fast and fast.next:                  # SAI: phải so với end
            slow = slow.next
            fast = fast.next.next
        root = TreeNode(slow.val)
        if slow is not start:
            root.left = build(start, slow)
        if slow.next is not end:
            root.right = build(slow.next, end)
        return root
    return build(head, None)

try:
    r = wrong_fast_slow(build_list([1, 2, 3, 4, 5, 6, 7]))
    print("Sai :", serialize(r))
except RecursionError:
    print("Sai : RecursionError — đệ quy không bao giờ dừng")

Sai : RecursionError — đệ quy không bao giờ dừng


## Lỗi 4: Cắt danh sách bằng cách sửa `next` rồi không khôi phục

Một số lời giải đặt `prev.next = None` để tách nửa trái — được chấp nhận trên LeetCode, nhưng **phá hủy** danh sách đầu vào. Cách nửa mở `[start, end)` tránh được.

# 12. Mẹo nhận diện pattern

- **Dãy đã sắp xếp → BST cân bằng**: chọn giữa làm gốc (#108 với mảng, #109 với danh sách, #1382 Balance a BST).
- **Dựng cây theo thứ tự inorder** khi dữ liệu đến tuần tự (luồng, danh sách liên kết): đệ quy với con trỏ chung, dựng trái → gốc → phải.
- **Tìm giữa danh sách liên kết**: nhanh/chậm (#876 Middle of the Linked List).

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với danh sách `1 → 2 → 3 → 4 → 5 → 6`, vẽ cây mà cách 3 dựng ra.

## Bài 2 — Điền chỗ trống (cách 3)

In [21]:
def exercise_2_sorted_list_to_bst(head: Optional[ListNode]) -> Optional[TreeNode]:
    n, node = 0, head
    while node:
        n += 1
        node = node.next
    cur = head
    def build(lo, hi):
        nonlocal cur
        # TODO: rỗng; mid; dựng trái; tạo gốc từ cur và tiến cur; dựng phải
        pass
    return build(0, n - 1)


# r = exercise_2_sorted_list_to_bst(build_list([-10, -3, 0, 5, 9])); assert valid_answer([-10, -3, 0, 5, 9], r)
# assert exercise_2_sorted_list_to_bst(build_list([])) is None

## Bài 3 — Biến thể: Convert Sorted Array to BST (LeetCode #108)

In [22]:
def exercise_3_sorted_array_to_bst(nums: List[int]) -> Optional[TreeNode]:
    # TODO
    pass


# assert valid_answer([-10, -3, 0, 5, 9], exercise_3_sorted_array_to_bst([-10, -3, 0, 5, 9]))
# assert valid_answer([1, 3], exercise_3_sorted_array_to_bst([1, 3]))

## Bài 4 — Biến thể: Balance a Binary Search Tree (LeetCode #1382)

Cho một BST bất kỳ (có thể rất lệch), trả về BST **cân bằng** chứa cùng các giá trị.

Gợi ý nhanh: inorder → mảng đã sắp xếp → Bài 3.

In [23]:
def exercise_4_balance_bst(root: Optional[TreeNode]) -> Optional[TreeNode]:
    # TODO
    pass


# r = exercise_4_balance_bst(wrong_insert_sorted(build_list([1, 2, 3, 4, 5, 6, 7]))); assert valid_answer([1, 2, 3, 4, 5, 6, 7], r)

## Bài 5 — Mini challenge: chuyển ngược BST → danh sách liên kết đôi vòng (LeetCode #426)

Biến BST thành danh sách **liên kết đôi, vòng, đã sắp xếp** **tại chỗ**: dùng `left` làm `prev` và `right` làm `next`. Trả về nút nhỏ nhất.

Gợi ý nhanh: duyệt inorder, giữ `prev`; nối `prev.right = node`, `node.left = prev`; cuối cùng nối đầu và cuối.

In [24]:
def exercise_5_tree_to_doubly_list(root: Optional[TreeNode]) -> Optional[TreeNode]:
    # TODO
    pass


# h = exercise_5_tree_to_doubly_list(to_bst_array(build_list([1, 2, 3, 4, 5]))); vals = [];  x = h
# for _ in range(5): vals.append(x.val); x = x.right
# assert vals == [1, 2, 3, 4, 5] and x is h and h.left.val == 5

# 14. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_inorder_build([1, 2, 3, 4, 5, 6])` sau khi tự làm.

**Bài 3**
- Mức 1: chính là `build(lo, hi)` của cách 1.

**Bài 4**
- Mức 1: `vals = inorder_of(root)`; rồi Bài 3.

**Bài 5**
- Mức 1: hàm `dfs(node)` inorder; biến `first`, `last` (nonlocal).
- Mức 2: khi thăm `node`: nếu `last` có thì `last.right = node; node.left = last`, ngược lại `first = node`; `last = node`.
- Mức 3: cuối cùng `last.right = first; first.left = last`.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [25]:
_ = trace_inorder_build([1, 2, 3, 4, 5, 6])

build(0, 5): mid = 2, dựng trái trước
  build(0, 1): mid = 0, dựng trái trước
  -> con trỏ đang ở 1: tạo gốc 1 cho đoạn [0, 1]
    build(1, 1): mid = 1, dựng trái trước
    -> con trỏ đang ở 2: tạo gốc 2 cho đoạn [1, 1]
-> con trỏ đang ở 3: tạo gốc 3 cho đoạn [0, 5]
  build(3, 5): mid = 4, dựng trái trước
    build(3, 3): mid = 3, dựng trái trước
    -> con trỏ đang ở 4: tạo gốc 4 cho đoạn [3, 3]
  -> con trỏ đang ở 5: tạo gốc 5 cho đoạn [3, 5]
    build(5, 5): mid = 5, dựng trái trước
    -> con trỏ đang ở 6: tạo gốc 6 cho đoạn [5, 5]
Cây: [3, 1, 5, None, 2, 4, 6]


## Đáp án Bài 2

In [26]:
def answer_2_sorted_list_to_bst(head: Optional[ListNode]) -> Optional[TreeNode]:
    return to_bst_inorder(head)

r = answer_2_sorted_list_to_bst(build_list([-10, -3, 0, 5, 9])); assert valid_answer([-10, -3, 0, 5, 9], r)
assert answer_2_sorted_list_to_bst(build_list([])) is None
print("Bài 2 PASS (giống to_bst_inorder ở mục 7)")

Bài 2 PASS (giống to_bst_inorder ở mục 7)


## Đáp án Bài 3

In [27]:
def answer_3_sorted_array_to_bst(nums: List[int]) -> Optional[TreeNode]:
    def build(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        return TreeNode(nums[mid], build(lo, mid - 1), build(mid + 1, hi))
    return build(0, len(nums) - 1)

assert valid_answer([-10, -3, 0, 5, 9], answer_3_sorted_array_to_bst([-10, -3, 0, 5, 9]))
assert valid_answer([1, 3], answer_3_sorted_array_to_bst([1, 3]))
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [28]:
def answer_4_balance_bst(root: Optional[TreeNode]) -> Optional[TreeNode]:
    return answer_3_sorted_array_to_bst(inorder_of(root))

r = answer_4_balance_bst(wrong_insert_sorted(build_list([1, 2, 3, 4, 5, 6, 7]))); assert valid_answer([1, 2, 3, 4, 5, 6, 7], r)
print("Bài 4 PASS | cây sau khi cân bằng:", serialize(r))

Bài 4 PASS | cây sau khi cân bằng: [4, 2, 6, 1, 3, 5, 7]


## Đáp án Bài 5

In [29]:
def answer_5_tree_to_doubly_list(root: Optional[TreeNode]) -> Optional[TreeNode]:
    if not root:
        return None
    first = last = None
    def dfs(node):
        nonlocal first, last
        if not node:
            return
        dfs(node.left)
        if last:
            last.right = node
            node.left = last
        else:
            first = node
        last = node
        dfs(node.right)
    dfs(root)
    last.right = first
    first.left = last
    return first

h = answer_5_tree_to_doubly_list(to_bst_array(build_list([1, 2, 3, 4, 5]))); vals = []; x = h
for _ in range(5):
    vals.append(x.val); x = x.right
assert vals == [1, 2, 3, 4, 5] and x is h and h.left.val == 5
print("Bài 5 PASS")

Bài 5 PASS


Ghi đè `last.right = node` là an toàn: khi `dfs` tới `node`, cây con phải của `last` đã được duyệt xong (vì `node` là nút liền sau `last` theo inorder), nên con trỏ `right` cũ của `last` không còn cần nữa.

# 16. Tổng kết

> Phần tử giữa làm gốc cho cây cân bằng. Trên danh sách liên kết: đổi sang mảng (`O(n)` bộ nhớ), nhanh/chậm (`O(n log n)`), hoặc tốt nhất là **dựng theo thứ tự inorder** với một con trỏ chạy dọc danh sách (`O(n)`, `O(log n)` bộ nhớ).

---

## Checklist trước khi nộp

- [ ] Chọn **giữa** làm gốc (không chèn tuần tự)?
- [ ] Cách inorder: dựng **trái trước**, rồi tạo gốc và tiến con trỏ?
- [ ] Nhanh/chậm: so với `end` của đoạn?
- [ ] Danh sách rỗng → `None`?